# Load Packages

In [1]:
import pandas as pd
import numpy as np
import pickle as pickle
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn import svm
from sklearn.metrics import matthews_corrcoef, make_scorer
from sklearn.model_selection import GridSearchCV

# Import Data

In [2]:
train_set = pd.read_csv("../data/clean/twitter_training_clean.csv", index_col=0)
test_set = pd.read_csv("../data/clean/twitter_test_clean.csv", index_col=0)
val_set = pd.read_csv("../data/clean/twitter_validation_clean.csv", index_col=0)

X_train = train_set.drop(labels="Sentiment", axis=1)
y_train = train_set["Sentiment"]

X_test = test_set.drop(labels="Sentiment", axis=1)
y_test = test_set["Sentiment"]

X_val = val_set.drop(labels="Sentiment", axis=1)
y_val = val_set["Sentiment"]

In [3]:
X_train

,Tweet Id,Entity,Tweet Content
49181,6122,FIFA,Last Weekend league for Fifa 20 Glad I could f...
43259,10298,PlayerUnknownsBattlegrounds(PUBG),omg i'm so excited to watch dk play pubg
55405,2398,CallOfDuty,all others who have problems with
14766,2956,Dota2,in
43611,10362,PlayerUnknownsBattlegrounds(PUBG),"minho, felix de jeongin sucked at pubg pretty ..."
...,...,...,...
37194,5237,Hearthstone,you
6265,289,Amazon,I'm not even going to show a 7-2 loss.
54886,2311,CallOfDuty,Fuck this call of duty update..
860,2553,Borderlands,I should get up & feed my dogs & such that way...


In [4]:
X_val

,Tweet Id,Entity,Tweet Content
60857,4926,GrandTheftAuto(GTA),Looks to me like he failed to check out the wa...
44454,11709,Verizon,"Wow, it takes all sorts of crazy people out th..."
72986,9020,Nvidia,Nvidia Unveils The World’s Fastest Gaming Moni...
36360,8295,Microsoft,Huge radio play here. Reinvention / Corporate ...
2299,1604,CallOfDutyBlackopsColdWar,SO I HAPPY WHO ABOUT THIS.
...,...,...,...
12476,8571,NBA2K,@Ronnie2K where is all my Mamba Edition extras...
49149,6117,FIFA,Sell 700k fifa coins fucking this game
12173,8517,NBA2K,@NBA2K $ 107 for a four game break and I can't...
4319,1949,CallOfDutyBlackopsColdWar,has called me a madman.. I understood right fr...


## Encoding

In [5]:
encoder = OrdinalEncoder()
encoder = encoder.fit(X_train)
X_train = encoder.transform(X_train)

encoder = encoder.fit(X_val)
X_val = encoder.transform(X_val)

encoder = encoder.fit(X_test)
X_test = encoder.transform(X_test)

In [6]:
X_train

array([[5.9300e+03, 1.0000e+01, 3.0056e+04],
       [1.0001e+04, 2.4000e+01, 5.3175e+04],
       [2.3370e+03, 6.0000e+00, 4.9602e+04],
       ...,
       [2.2500e+03, 6.0000e+00, 1.8167e+04],
       [2.4840e+03, 4.0000e+00, 2.4862e+04],
       [3.0470e+03, 9.0000e+00, 4.6437e+04]], shape=(59196, 3))

In [7]:
X_val

array([[3.4980e+03, 1.4000e+01, 7.8000e+03],
       [8.2960e+03, 2.8000e+01, 1.2301e+04],
       [6.4040e+03, 2.1000e+01, 8.6150e+03],
       ...,
       [6.0360e+03, 2.0000e+01, 1.7920e+03],
       [1.3890e+03, 7.0000e+00, 1.3116e+04],
       [7.5300e+03, 2.5000e+01, 9.2460e+03]], shape=(14800, 3))

In [8]:
X_test

array([[263.,  11., 419.],
       [ 28.,   0., 213.],
       [646.,  19.,  91.],
       ...,
       [205.,   4., 780.],
       [632.,  19., 228.],
       [542.,  31., 501.]], shape=(1000, 3))

## Scaling

In [9]:
scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train)
X_val = scaler.fit_transform(X_val)

In [10]:
X_train

array([[1.65263951, 1.08352099, 1.84175933],
       [2.78719186, 2.60045038, 3.25843599],
       [0.65130161, 0.65011259, 3.03949115],
       ...,
       [0.62705546, 0.65011259, 1.11323003],
       [0.69226923, 0.4334084 , 1.52348351],
       [0.84917244, 0.97516889, 2.84554757]], shape=(59196, 3))

In [11]:
X_val

array([[1.33514751, 1.51690283, 1.87125225],
       [3.16649049, 3.03380566, 2.95106076],
       [2.44433523, 2.27535424, 2.06677412],
       ...,
       [2.30387374, 2.16700404, 0.42990821],
       [0.53016578, 0.75845141, 3.14658263],
       [2.87411685, 2.70875505, 2.21815363]], shape=(14800, 3))

In [12]:
X_test

array([[263.,  11., 419.],
       [ 28.,   0., 213.],
       [646.,  19.,  91.],
       ...,
       [205.,   4., 780.],
       [632.,  19., 228.],
       [542.,  31., 501.]], shape=(1000, 3))

# Model

In [13]:
svc = svm.LinearSVC(random_state=42)
matthews_scorer = make_scorer(matthews_corrcoef)

## Hyperparameter Search

In [14]:
param_grid = {
    "loss": ["hinge", "squared_hinge"],
    "dual": [True, False],
    "C": [0.01, 0.1, 0.5, 1.0],
    "max_iter": [500, 1000, 2000, 5000, 10000, 20000]
}

grid_search = GridSearchCV(svc, param_grid, scoring=matthews_scorer, verbose=3)
grid_search.fit(X_train, y_train)

grid_search.cv_results_

Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV 1/5] END C=0.01, dual=True, loss=hinge, max_iter=500;, score=-0.003 total time=   0.1s
[CV 2/5] END C=0.01, dual=True, loss=hinge, max_iter=500;, score=-0.026 total time=   0.1s
[CV 3/5] END C=0.01, dual=True, loss=hinge, max_iter=500;, score=-0.027 total time=   0.1s
[CV 4/5] END C=0.01, dual=True, loss=hinge, max_iter=500;, score=-0.000 total time=   0.1s
[CV 5/5] END C=0.01, dual=True, loss=hinge, max_iter=500;, score=0.041 total time=   0.1s
[CV 1/5] END C=0.01, dual=True, loss=hinge, max_iter=1000;, score=-0.003 total time=   0.1s
[CV 2/5] END C=0.01, dual=True, loss=hinge, max_iter=1000;, score=-0.026 total time=   0.1s
[CV 3/5] END C=0.01, dual=True, loss=hinge, max_iter=1000;, score=-0.027 total time=   0.1s
[CV 4/5] END C=0.01, dual=True, loss=hinge, max_iter=1000;, score=-0.000 total time=   0.1s
[CV 5/5] END C=0.01, dual=True, loss=hinge, max_iter=1000;, score=0.041 total time=   0.1s
[CV 1/5] END C=0.01, dual

C:\Users\ttjaa\Documents\College Part II\CSCI-635\csci-635-project\.venv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 5/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=500;, score=0.057 total time=   6.9s
[CV 1/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=1000;, score=0.060 total time=   5.7s
[CV 2/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=1000;, score=0.046 total time=   6.2s
[CV 3/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=1000;, score=0.056 total time=   6.1s
[CV 4/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=1000;, score=0.041 total time=   6.2s
[CV 5/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=1000;, score=0.057 total time=   9.2s
[CV 1/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=2000;, score=0.060 total time=   7.1s
[CV 2/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=2000;, score=0.046 total time=   6.9s
[CV 3/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=2000;, score=0.056 total time=   6.7s
[CV 4/5] END C=0.5, dual=True, loss=squared_hinge, max_iter=2000;, score=0.041 total time=   8.1s
[CV 5/5] END C=0.5, d

C:\Users\ttjaa\Documents\College Part II\CSCI-635\csci-635-project\.venv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 1/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=500;, score=0.060 total time=  11.8s


...

[CV 1/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=1000;, score=0.060 total time=  29.3s
[CV 2/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=1000;, score=0.046 total time=  15.0s
[CV 3/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=1000;, score=0.056 total time=  16.6s
[CV 4/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=1000;, score=0.041 total time=  15.4s


C:\Users\ttjaa\Documents\College Part II\CSCI-635\csci-635-project\.venv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 5/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=1000;, score=0.057 total time=  19.3s
[CV 1/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=2000;, score=0.060 total time=  19.4s
[CV 2/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=2000;, score=0.046 total time=  14.3s
[CV 3/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=2000;, score=0.056 total time=  15.3s
[CV 4/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=2000;, score=0.041 total time=  16.6s
[CV 5/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=2000;, score=0.057 total time=  20.7s
[CV 1/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=5000;, score=0.060 total time=  14.8s
[CV 2/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=5000;, score=0.046 total time=  16.1s
[CV 3/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=5000;, score=0.056 total time=  14.7s
[CV 4/5] END C=1.0, dual=True, loss=squared_hinge, max_iter=5000;, score=0.041 total time=  16.3s
[CV 5/5] END C=1.0, 

C:\Users\ttjaa\Documents\College Part II\CSCI-635\csci-635-project\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
120 fits failed out of a total of 480.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
120 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\ttjaa\Documents\College Part II\CSCI-635\csci-635-project\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\ttjaa\Documents\College Part II\CSCI-635\csci-635-project\.venv\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^

{'mean_fit_time': array([ 0.18251085,  0.17402658,  0.18319654,  0.17815652,  0.13453617,
         0.13449259,  0.51082768,  0.56693425,  0.57921791,  0.57556009,
         0.50277276,  0.40354838,  0.08959341,  0.0586442 ,  0.06065884,
         0.06040735,  0.06677904,  0.08084221,  0.2513762 ,  0.35446897,
         0.31160765,  0.32524681,  0.62368255,  0.33240452,  0.26321478,
         0.2217042 ,  0.25761085,  0.25701556,  0.203405  ,  0.217026  ,
         2.27269869,  1.3609653 ,  1.26434579,  1.45109696,  1.34618988,
         1.39841967,  0.05766263,  0.0716476 ,  0.0682313 ,  0.05718679,
         0.06676083,  0.06094604,  0.18038425,  0.17460356,  0.18092327,
         0.20392094,  0.18725009,  0.17704091,  0.37878833,  0.31396055,
         0.392068  ,  0.44370542,  0.47357335,  0.48831244,  6.4899004 ,
         6.75332723,  9.39240265, 11.96776533,  9.49727311,  8.71594105,
         0.05773339,  0.05508647,  0.06013355,  0.06800799,  0.06140203,
         0.05850687,  0.21949768, 

In [15]:
best_svc = grid_search.best_estimator_

## Training

In [16]:
train_score = best_svc.score(X_train, y_train)

In [17]:
print("Training Score: ", train_score)

Training Score:  0.3239576998445841


## Validation

In [18]:
val_score = best_svc.score(X_val, y_val)

print("Validation Score: ", val_score)

Validation Score:  0.3170945945945946


## Test

In [19]:
test_score = best_svc.score(X_test, y_test)

print("Test Score: ", test_score)

Test Score:  0.277
